# TinyDoc-VLM — Fast LoRA Training on Colab Free T4

This notebook fine-tunes TinyDoc-VLM using LoRA on a free Colab T4 GPU.

**Expected time:** ~1-2 hours for 5K steps on T4

In [ ]:
#@title 1. Install Dependencies (~2 min)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate peft datasets pillow faker augraphy albumentations sentencepiece

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
#@title 2. Clone Repo & Install Package
!git clone https://github.com/eulogik/TinyDoc-VLM.git
%cd TinyDoc-VLM
!pip install -e .

In [ ]:
#@title 3. Generate 500 Synthetic Documents (~1 min)
!python data/synthetic/generator.py --num-docs 500 --output-dir data/synthetic/colab_output

In [ ]:
#@title 4. Train with LoRA (5K steps, ~1 hour on T4)
!python training/fast_train.py \
    --manifest data/synthetic/colab_output/manifest.jsonl \
    --data-root data/synthetic \
    --steps 5000 \
    --batch-size 4 \
    --grad-accum 4 \
    --lr 2e-4 \
    --warmup 200 \
    --max-samples 50000 \
    --lora-rank 16 \
    --output-dir checkpoints/lora

In [ ]:
#@title 5. Upload to HuggingFace Hub
from huggingface_hub import HfApi, login

#@markdown Enter your HuggingFace token:
HF_TOKEN = "hf_YOUR_TOKEN_HERE"  #@param {type:"string"}

login(token=HF_TOKEN)
api = HfApi()

repo_id = "eulogik/TinyDoc-VLM-LoRA"
try:
    api.create_repo(repo_id, exist_ok=True)
    print(f"Created repo: {repo_id}")
except Exception as e:
    print(f"Repo exists or error: {e}")

api.upload_folder(
    folder_path="checkpoints/lora/final",
    repo_id=repo_id,
    repo_type="model",
)
print(f"Uploaded to: https://huggingface.co/{repo_id}")

In [ ]:
#@title 6. Quick Eval (5 samples)
!python training/eval_lora.py \
    --checkpoint checkpoints/lora/final \
    --benchmark ocrbench \
    --max-samples 5 \
    --device cuda